# Assignment 04: Custom Dataset (100 points)

**Unit 06: Programming PyTorch | AI 310**

USAAIO Round 2 provides data in custom formats. You must be fluent in wrapping any data source into a PyTorch `Dataset` and feeding it through a `DataLoader`. This assignment covers the Dataset protocol, custom collation, and data pipeline construction.

**Notation**:
- $N$ = number of samples, $B$ = batch size
- `Dataset.__getitem__` returns a single sample
- `DataLoader` batches samples automatically

In [ ]:
"""DO NOT MAKE ANY CHANGE IN THIS CELL."""
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import Dataset, DataLoader

torch.manual_seed(42)
np.random.seed(42)

**WARNING**: Do not import any additional libraries beyond those provided above.

---

## Part 1 (15 points, coding)

Implement a `SyntheticClassificationDataset` that generates random classification data.

- Features: `n_samples` points in $\mathbb{R}^{n\_features}$ drawn from $\mathcal{N}(0, 1)$
- Labels: random integers in $[0, n\_classes)$
- Features should be `float32`, labels should be `long`
- Data should be generated in `__init__` (not on-the-fly)

In [ ]:
### WRITE YOUR SOLUTION HERE ###

class SyntheticClassificationDataset(Dataset):
    def __init__(self, n_samples, n_features, n_classes):
        pass
    
    def __len__(self):
        pass
    
    def __getitem__(self, idx):
        # Return (feature_vector, label)
        pass

In [ ]:
""" END OF THIS PART """
ds = SyntheticClassificationDataset(1000, 20, 5)
assert len(ds) == 1000
x, y = ds[0]
assert x.shape == (20,) and x.dtype == torch.float32
assert y.dtype == torch.long and 0 <= y.item() < 5
# Test with DataLoader
loader = DataLoader(ds, batch_size=32, shuffle=True)
batch_x, batch_y = next(iter(loader))
assert batch_x.shape == (32, 20)
assert batch_y.shape == (32,)
print("Part 1 passed!")

---

## Part 2 (20 points, coding)

Implement a `PolynomialRegressionDataset` that generates data from a noisy polynomial:

$$y = a_0 + a_1 x + a_2 x^2 + \cdots + a_d x^d + \epsilon$$

where $\epsilon \sim \mathcal{N}(0, \text{noise\_std}^2)$.

Requirements:
- Accept `coefficients` (list of floats $[a_0, a_1, \ldots, a_d]$), `n_samples`, `x_range` (tuple), and `noise_std`
- Features: the full feature vector $[1, x, x^2, \ldots, x^d]$ of shape `(d+1,)` for each sample
- Labels: the scalar $y$ value of shape `(1,)`
- $x$ values should be uniformly sampled from `x_range`

In [ ]:
### WRITE YOUR SOLUTION HERE ###

class PolynomialRegressionDataset(Dataset):
    def __init__(self, coefficients, n_samples=500, x_range=(-3, 3), noise_std=0.1):
        """
        Args:
            coefficients: [a_0, a_1, ..., a_d] polynomial coefficients
            n_samples: number of data points
            x_range: (min, max) for sampling x
            noise_std: standard deviation of Gaussian noise
        """
        pass
    
    def __len__(self):
        pass
    
    def __getitem__(self, idx):
        # Return (features, target) where features = [1, x, x^2, ...], target = [y]
        pass

In [ ]:
""" END OF THIS PART """
# y = 1 + 2x + 3x^2 (quadratic)
ds = PolynomialRegressionDataset([1.0, 2.0, 3.0], n_samples=200, noise_std=0.0)
assert len(ds) == 200
feat, target = ds[0]
assert feat.shape == (3,), f"Feature shape should be (3,), got {feat.shape}"  # [1, x, x^2]
assert target.shape == (1,), f"Target shape should be (1,), got {target.shape}"
# With zero noise, y should exactly equal a0 + a1*x + a2*x^2
x_val = feat[1].item()  # x
expected_y = 1.0 + 2.0 * x_val + 3.0 * x_val ** 2
assert abs(target.item() - expected_y) < 1e-4, f"Expected {expected_y}, got {target.item()}"
print("Part 2 passed!")

---

## Part 3 (20 points, coding)

Implement a **custom collate function** for variable-length graph data.

Each sample is a tuple `(node_features, adjacency_matrix, label)` where:
- `node_features`: shape `(num_nodes, feat_dim)` — different number of nodes per graph
- `adjacency_matrix`: shape `(num_nodes, num_nodes)` — square matrix
- `label`: scalar integer

Your collate function must:
1. Pad `node_features` to the max number of nodes in the batch → `(B, max_nodes, feat_dim)`
2. Pad `adjacency_matrix` to `(B, max_nodes, max_nodes)`
3. Create a `node_mask` of shape `(B, max_nodes)` where `True` means the node is real (not padding)
4. Stack labels into `(B,)`

In [ ]:
### WRITE YOUR SOLUTION HERE ###

def graph_collate_fn(batch):
    """
    Custom collate for variable-size graphs.
    
    Args:
        batch: list of (node_features, adj_matrix, label) tuples
    
    Returns:
        padded_features: (B, max_nodes, feat_dim)
        padded_adj: (B, max_nodes, max_nodes)
        node_mask: (B, max_nodes) bool, True = real node
        labels: (B,)
    """
    pass

In [ ]:
""" END OF THIS PART """
# Create test data: 3 graphs with different sizes
feat_dim = 8
samples = [
    (torch.randn(3, feat_dim), torch.randint(0, 2, (3, 3)).float(), 0),
    (torch.randn(5, feat_dim), torch.randint(0, 2, (5, 5)).float(), 1),
    (torch.randn(2, feat_dim), torch.randint(0, 2, (2, 2)).float(), 0),
]

features, adj, mask, labels = graph_collate_fn(samples)

assert features.shape == (3, 5, feat_dim), f"Expected (3, 5, 8), got {features.shape}"
assert adj.shape == (3, 5, 5), f"Expected (3, 5, 5), got {adj.shape}"
assert mask.shape == (3, 5), f"Expected (3, 5), got {mask.shape}"
assert labels.shape == (3,)

# Check mask: graph 0 has 3 nodes, graph 1 has 5, graph 2 has 2
assert mask[0].sum() == 3
assert mask[1].sum() == 5
assert mask[2].sum() == 2

# Check padding is zero
assert features[0, 3:].sum() == 0, "Padding should be zero"
assert adj[2, 2:, :].sum() == 0, "Padding in adj should be zero"
print("Part 3 passed!")

---

## Part 4 (20 points, coding)

Implement a `TransformDataset` wrapper that applies a chain of transformations to samples from a base dataset.

Requirements:
- Accept a `base_dataset` and a list of `transforms` (each is a callable)
- In `__getitem__`, fetch the sample from the base dataset, then apply each transform in order to the features (not the label)
- `__len__` should match the base dataset

In [ ]:
### WRITE YOUR SOLUTION HERE ###

class TransformDataset(Dataset):
    def __init__(self, base_dataset, transforms):
        """
        Args:
            base_dataset: Dataset returning (feature, label) tuples
            transforms: list of callables, each takes and returns a tensor
        """
        pass
    
    def __len__(self):
        pass
    
    def __getitem__(self, idx):
        # Fetch from base, apply transforms to feature, return (transformed_feature, label)
        pass

In [ ]:
""" END OF THIS PART """
base = SyntheticClassificationDataset(100, 20, 5)

# Define transforms
normalize = lambda x: (x - x.mean()) / (x.std() + 1e-8)
add_noise = lambda x: x + 0.01 * torch.randn_like(x)

transformed = TransformDataset(base, [normalize, add_noise])
assert len(transformed) == 100

x_orig, y_orig = base[0]
x_trans, y_trans = transformed[0]

assert x_trans.shape == x_orig.shape
assert y_trans == y_orig, "Label should not be transformed"
# After normalization, mean should be ~0
x_normed = normalize(x_orig)
assert abs(x_normed.mean().item()) < 1e-5, "Normalization not applied correctly"
print("Part 4 passed!")

---

## Part 5 (25 points, coding)

Build a complete **data pipeline** and verify it works end-to-end with a model.

1. Create a `SyntheticClassificationDataset` with 1000 samples, 50 features, 3 classes
2. Split into 80% train / 20% validation using `torch.utils.data.random_split`
3. Create `DataLoader`s with batch_size=32, shuffle=True for train, False for val
4. Create a simple model: `nn.Linear(50, 3)`
5. Run ONE epoch of training (forward, loss, backward, step) and ONE epoch of validation
6. Store the final training loss as `final_train_loss` and validation accuracy as `final_val_acc`

In [ ]:
### WRITE YOUR SOLUTION HERE ###

torch.manual_seed(42)

# 1. Create dataset
# 2. Split into train/val
# 3. Create DataLoaders
# 4. Create model, optimizer, criterion
# 5. Train for one epoch
# 6. Evaluate on validation set

final_train_loss = ...  # scalar float
final_val_acc = ...     # scalar float between 0 and 1

In [ ]:
""" END OF THIS PART """
assert isinstance(final_train_loss, float), "final_train_loss must be a float"
assert isinstance(final_val_acc, float), "final_val_acc must be a float"
assert 0 <= final_val_acc <= 1, f"Accuracy must be in [0, 1], got {final_val_acc}"
assert final_train_loss > 0, "Loss must be positive"
print(f"Part 5 passed! Train loss: {final_train_loss:.4f}, Val acc: {final_val_acc:.4f}")